# NARX maneuver and Gateway tracking results

The uncontrolled halo observer is used to track Lunar Gateway. The first dataset computes correction labels but does not execute them. The phase-aware dataset evaluates possible burn times. The closed-loop dataset applies selected maneuvers and continues the LSO and Gateway optical tracking filters.

The primary result is how accurately Gateway is tracked **after the maneuvers are applied**, measured against the 100 m position and 0.01 m/s velocity targets. The plots below separate this from how well a model reproduces the guidance labels.

In [1]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "configs/baseline.json").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook from the repository checkout")
preview = ROOT / "results/narx_closed_loop_preview"
full = ROOT / "results/narx_closed_loop"
dataset_dir = full if (full / "test.npz").exists() else preview
if not (dataset_dir / "test.npz").exists():
    display(Markdown("**No closed-loop results found.** Run the main notebook or `python scripts/run_closed_loop_narx.py`, then rerun this notebook."))
else:
    display(Markdown(f"Reading **{dataset_dir.relative_to(ROOT)}**. " + ("Full Monte Carlo dataset." if dataset_dir == full else "Small notebook preview, **one arc per split**; do not interpret as Monte Carlo validation.")))

**No closed-loop results found.** Run the main notebook or `python scripts/run_closed_loop_narx.py`, then rerun this notebook.

## Gateway tracking after maneuvers

Position and velocity errors are calculated from the estimated and simulated true Gateway states:

$$
e_r(t)=\|\hat{\mathbf r}_{G}(t)-\mathbf r_G(t)\|,\qquad
e_v(t)=\|\hat{\mathbf v}_{G}(t)-\mathbf v_G(t)\|.
$$

The 100 m and 0.01 m/s limits must be satisfied **at the same time**. The first plot shows error histories; the table reports how often both limits are met. A successful result requires consistent performance across independent Monte Carlo realizations, not a single favorable trajectory.

In [2]:
if (dataset_dir / "test.npz").exists():
    with np.load(dataset_dir / "test.npz") as d:
        t = np.array(d["decision_times_tu"]) * 375190.25852 / 86400
        p = np.array(d["gateway_tracking_position_error_km"])
        v = np.array(d["gateway_tracking_velocity_error_m_s"])
        both = (p <= 0.1) & (v <= 0.01)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
        for i in range(min(len(p), 8)):
            ax1.plot(t, p[i]*1000, alpha=.55, linewidth=1)
            ax2.plot(t, v[i], alpha=.55, linewidth=1)
        ax1.axhline(100, linestyle="--", color="black", label="100 m target")
        ax2.axhline(.01, linestyle="--", color="black", label="0.01 m/s target")
        ax1.set_ylabel("Gateway position error (m)")
        ax2.set_ylabel("Gateway velocity error (m/s)")
        ax2.set_xlabel("Time (days)")
        ax1.legend()
        ax2.legend()
        ax1.set_yscale("symlog", linthresh=1)
        ax2.set_yscale("symlog", linthresh=.001)
        fig.suptitle("Gateway tracking accuracy during applied-burn simulation")
        plt.tight_layout()
        plt.show()
        display(Markdown(
            f"**Held-out test:** {len(p)} episode(s). Position within 100 m: **{100*np.mean(p<=.1):.1f}%** of decision epochs. "
            f"Velocity within 0.01 m/s: **{100*np.mean(v<=.01):.1f}%**. "
            f"**Both limits simultaneously: {100*np.mean(both):.1f}%**. "
            "These are tracking-error metrics, not reference-departure metrics."
        ))

## Maneuver labels versus executed burns

The STM-based guidance algorithm requests a correction when estimated departure crosses a trigger. A **valid label** can be a zero burn when no correction is needed. Therefore valid-label counts should not be read as successful maneuvers.

This section separates requested corrections, accepted corrections, rejected requests and actual executed impulses. A burn can be selected at one decision and executed between two decisions, so the saved applied-burn history must be interpreted with its recorded execution time.

In [3]:
if (dataset_dir / "test.npz").exists():
    with np.load(dataset_dir / "test.npz") as d:
        lines = ["| Spacecraft | Decisions | Requested | Feasible requested | Rejected requested | Nonzero applied-burn records |", "|---|---:|---:|---:|---:|---:|"]
        for name in ("observer", "gateway"):
            requested = np.asarray(d[name + "_maneuver_required"], dtype=bool)
            valid = np.asarray(d[name + "_maneuver_valid"], dtype=bool)
            applied = np.linalg.norm(d[name + "_applied_delta_v_rotating_m_s"], axis=-1) > 0
            lines.append(f"| {name.title()} | {requested.size} | {requested.sum()} | {(requested & valid).sum()} | {(requested & ~valid).sum()} | {applied.sum()} |")
        display(Markdown("\n".join(lines)))
        display(Markdown("The applied-burn count reflects **saved nonzero records**, not necessarily distinct commands when multiple impulses occur between decision epochs."))

## Two-stage learned maneuver predictor

The first model predicts **maneuver or no maneuver**. For requested feasible burns, the next models predict the three components of Δv and one of four burn-delay classes. The classifier is evaluated with precision, recall and F1. The Δv predictor uses RMSE in m/s, and the burn-delay classifier reports accuracy.

The learned-policy metrics measure **imitation of the STM guidance labels**, not whether predicted maneuvers maintain Gateway tracking. A learned closed-loop evaluation is needed before claiming tracking performance from NARX.

In [4]:
metrics_path = ROOT / "results/narx_two_stage/metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    lines = ["| Spacecraft | Split | F1 | Precision | Recall | Δv RMSE (m/s) | Delay accuracy |", "|---|---|---:|---:|---:|---:|---:|"]
    for name, result in metrics.items():
        for split in ("validation", "test"):
            m = result.get(split)
            if m is None:
                continue
            rmse = "N/A" if m["delta_v_rmse_m_s"] is None else f'{m["delta_v_rmse_m_s"]:.5f}'
            delay = "N/A" if m["burn_delay_class_accuracy_teacher_forced"] is None else f'{100*m["burn_delay_class_accuracy_teacher_forced"]:.1f}%'
            lines.append(f'| {name.title()} | {split} | {m["maneuver_f1"]:.3f} | {m["maneuver_precision"]:.3f} | {m["maneuver_recall"]:.3f} | {rmse} | {delay} |')
    display(Markdown("\n".join(lines)))
else:
    display(Markdown("**Learned-model scores have not been generated in this checkout.** Run `python scripts/run_two_stage_narx.py` to generate Monte Carlo data and evaluate the two-stage predictor, then rerun this notebook. No numerical learning performance is assumed."))

**Learned-model scores have not been generated in this checkout.** Run `python scripts/run_two_stage_narx.py` to generate Monte Carlo data and evaluate the two-stage predictor, then rerun this notebook. No numerical learning performance is assumed.